In [ ]:
# hide
import numpy as np
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# hide
def pluck(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A Karplus-Strong plucked string (Chapter 12) that decays by 60 dB in `duration`
    seconds. An all-pass filter (Chapter 13) in the loop tunes it precisely."""
    P = f_s / pq.helper.pitch_to_frequency(pitch)   # the period, in (fractional) samples
    M = int(P - 0.6)                                 # the delay line (the low-pass adds 1/2 sample)
    D = P - 0.5 - M                                  # the all-pass adds the remaining D samples
    c = (1 - D) / (1 + D)                            # the all-pass gain for a delay of D samples
    g = 10 ** (-3 * P / (f_s * duration))            # decay by 60 dB over the duration
    x = np.zeros(int(duration * f_s))
    x[:M] = np.random.uniform(-1.0, 1.0, M)          # excitation: a burst of white noise
    # The whole loop is one recursive filter (Chapter 9), which SciPy runs quickly:
    # y[n] = x[n] + c x[n-1] - c y[n-1] + (g/2) (c y[n-M] + (1 + c) y[n-M-1] + y[n-M-2])
    a = np.zeros(M + 3)
    a[:2] = [1.0, c]
    a[M:] -= 0.5 * g * np.array([c, 1 + c, 1.0])
    return pq.Audio(lfilter([1.0, c], a, x), f_s)

In [ ]:
def markov(N: int, states: list[int], initial_state: int | None, probabilities: np.ndarray,
           ioi: float = 0.25, duration: float = 0.5) -> pq.Score:
    """A melody of N notes from a Markov chain. Row a of `probabilities` is the
    distribution of the next state, given that the previous state is states[a].
    Onsets are in beats, and durations (decay times) are in seconds."""
    if initial_state is None:
        a = np.random.randint(len(states))           # the first state, uniformly at random
    else:
        a = states.index(initial_state)
    events = []
    for i in range(N):
        events.append((i * ioi, {"pitch": states[a], "duration": duration}))
        a = np.random.choice(len(states), p=probabilities[a])   # transition to the next state
    return pq.Score(events)


#         C4  D4  E4  F4  G4  A4  B4  C5  D5  E5  F5  G5
states = [60, 62, 64, 65, 67, 69, 71, 72, 74, 76, 77, 79]
probabilities = np.array([
    # C4   D4   E4   F4   G4   A4   B4   C5   D5   E5   F5   G5
    [0.0, 0.4, 0.3, 0.0, 0.2, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0],  # from C4
    [0.4, 0.0, 0.4, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # from D4
    [0.2, 0.3, 0.0, 0.3, 0.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # from E4
    [0.0, 0.1, 0.5, 0.0, 0.3, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # from F4
    [0.0, 0.0, 0.2, 0.2, 0.1, 0.3, 0.0, 0.2, 0.0, 0.0, 0.0, 0.0],  # from G4
    [0.0, 0.0, 0.0, 0.1, 0.4, 0.0, 0.3, 0.2, 0.0, 0.0, 0.0, 0.0],  # from A4
    [0.0, 0.0, 0.0, 0.0, 0.1, 0.2, 0.0, 0.7, 0.0, 0.0, 0.0, 0.0],  # from B4
    [0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.3, 0.1, 0.3, 0.1, 0.0, 0.0],  # from C5
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.4, 0.0, 0.4, 0.1, 0.0],  # from D5
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.3, 0.0, 0.3, 0.2],  # from E5
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.6, 0.0, 0.2],  # from F5
    [0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.0, 0.2, 0.0, 0.3, 0.3, 0.0],  # from G5
])
assert np.allclose(probabilities.sum(axis=1), 1.0)   # each row is a distribution

score = markov(32, states, None, probabilities)
pq.play(score.render(pluck, metronome=pq.BasicMetronome(bpm=120)), normalize=True)